In [1]:
import torch
import torch.nn as nn
import json

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

from collections import Counter

In [2]:
BATCH_SIZE = 8
EPOCHS = 3
LR = 0.00005

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using:", DEVICE)

Using: cuda


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.RandomRotation(12),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.10,0.10),
        scale=(0.85,1.15)
    ),

    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.15
    ),

    transforms.RandomPerspective(
        distortion_scale=0.2,
        p=0.5
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
train_dataset = datasets.ImageFolder(
    "../dataset/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "../dataset/val",
    transform=val_transform
)

NUM_CLASSES = len(train_dataset.classes)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [7]:
from collections import Counter

targets = train_dataset.targets

class_counts = Counter(targets)

weights = []

for i in range(NUM_CLASSES):
    weights.append(
        len(targets) / class_counts[i]
    )

weights = torch.tensor(
    weights,
    dtype=torch.float32
)

weights = weights / weights.sum() * NUM_CLASSES

weights = weights.to(DEVICE)

print("Class Weights:")
print(weights)

Class Weights:
tensor([0.5715, 0.8939, 1.3557, 1.0001, 1.1789], device='cuda:0')


In [8]:
weights_pretrained = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(
    weights=weights_pretrained
)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

model.load_state_dict(
    torch.load(
        "best_document_classifier_finetuned.pth",
        map_location=DEVICE
    )
)

model = model.to(DEVICE)

print("Existing model loaded successfully.")

C:\Users\nitis\AppData\Local\Temp\ipykernel_25228\3321094436.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Existing model loaded successfully.


In [9]:
criterion = nn.CrossEntropyLoss(
    weight=weights
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

current_acc = 100 * correct / total

print(f"Current Model Validation Accuracy: {current_acc:.2f}%")

Current Model Validation Accuracy: 99.70%


In [11]:
#validation accuracy of current model
best_acc = current_acc  

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # Validation
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" | Loss: {avg_loss:.4f}"
        f" | Val Acc: {accuracy:.2f}%"
    )

    # Save only if improved
    if accuracy > best_acc:  

        best_acc = accuracy

        torch.save(
            model.state_dict(),
            "best_document_classifier_v2.pth"
        )

        print(
            f"✅ New Best Fine-Tuned Model Saved! ({accuracy:.2f}%)"
        )

    else:

        print(
            f"No improvement. Best Accuracy: {best_acc:.2f}%"
        )

print(f"\nFine-tuning completed. Best Validation Accuracy: {best_acc:.2f}%")

Epoch 1/3 | Loss: 0.0233 | Val Acc: 100.00%
✅ New Best Fine-Tuned Model Saved! (100.00%)
Epoch 2/3 | Loss: 0.0179 | Val Acc: 100.00%
No improvement. Best Accuracy: 100.00%
Epoch 3/3 | Loss: 0.0151 | Val Acc: 100.00%
No improvement. Best Accuracy: 100.00%

Fine-tuning completed. Best Validation Accuracy: 100.00%
